# UDA-Hub Agentic App

Entrypoint for the compiled multi-agent ticket-resolution graph. See
`agentic/design/architecture.md` for the full architecture writeup, and
`agentic/workflow.py` for the hand-built `StateGraph` this notebook drives
(no `create_react_agent` / `langgraph_supervisor` prebuilt helpers used).

**How to run**: install `solution/requirements.txt` into the project venv,
put a real `OPENAI_API_KEY` (and `OPENAI_BASE_URL` if using a proxy) in
`solution/.env` -- never commit this file -- then run all cells top to
bottom. This notebook demonstrates the system two ways: an interactive
REPL you can run yourself (documented, not executed here since it needs
live keyboard input), and a set of scripted end-to-end scenarios that *are*
executed and saved below, so every classification/routing/tool-use/outcome
decision is directly inspectable without having to re-run anything.

## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

Agents live under `agentic/agents/`, tools under `agentic/tools/`, and the
orchestration graph is assembled in `agentic/workflow.py`.

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

### Interactive usage (for manual testing)

`chat_interface()` opens a REPL against the real orchestrator -- each line
you type is run through the whole classify → route → resolve/escalate →
finalize pipeline, with `thread_id=ticket_id` keeping short-term memory
alive across turns. It isn't executed in this saved notebook (it blocks on
keyboard input, which a non-interactive run can't provide) -- run it
yourself in a live kernel:

```python
chat_interface(
    orchestrator,
    ticket_id="demo-1",
    account_id="cultpass",
    external_user_id="a4ab87",  # Alice Kingsley -- seeded blocked, always escalates
    channel="chat",
)
```

Type `quit` / `exit` / `q` to end the session.

### Scripted end-to-end demonstrations

Each `run_ticket(...)` call below drives the *real* compiled `orchestrator`
against the live model for one ticket, seeding a real `Ticket` row first if
one doesn't already exist so the Finalize step's database write actually
lands. It prints the resulting classification, route, every tool invoked
(name + success), confidence, final status, and the customer-facing
message -- covering classification, routing, knowledge retrieval, tool
usage, resolution attempts, and final action end to end.

In [4]:
import uuid

from langchain_core.messages import HumanMessage

from agentic.tools.db import get_udahub_engine
from data.models import udahub
from utils import get_session


def ensure_ticket(account_id, external_user_id, ticket_id, channel="chat", tags=""):
    """Create a Ticket + TicketMetadata row for this ticket_id if it doesn't
    already exist, creating the UDA-Hub User record for this customer too
    if this is their first ticket -- so Finalize's DB write has a real row
    to update, same as a genuine incoming ticket would."""
    engine = get_udahub_engine()
    with get_session(engine) as session:
        if session.get(udahub.Ticket, ticket_id) is not None:
            return
        user = (
            session.query(udahub.User)
            .filter_by(account_id=account_id, external_user_id=external_user_id)
            .one_or_none()
        )
        if user is None:
            user = udahub.User(
                user_id=str(uuid.uuid4()),
                account_id=account_id,
                external_user_id=external_user_id,
                user_name=external_user_id,
            )
            session.add(user)
            session.flush()
        session.add(udahub.Ticket(ticket_id=ticket_id, account_id=account_id, user_id=user.user_id, channel=channel))
        session.add(udahub.TicketMetadata(ticket_id=ticket_id, status="open", tags=tags))


def run_ticket(ticket_id, account_id, external_user_id, text, channel="chat", reported_urgency=None):
    ensure_ticket(account_id, external_user_id, ticket_id, channel=channel)
    trigger = {
        "messages": [HumanMessage(content=text)],
        "ticket_id": ticket_id,
        "account_id": account_id,
        "external_user_id": external_user_id,
        "channel": channel,
        "reported_urgency": reported_urgency,
        "ticket_text": text,
    }
    result = orchestrator.invoke(trigger, config={"configurable": {"thread_id": ticket_id}})

    cls = result.get("classification", {}) or {}
    trace = result.get("trace", [])
    routes = [e["route"] for e in trace if e.get("node") == "supervisor" and e.get("event") == "routed"]
    tool_calls = [
        f"{call.get('tool')}(ok={call.get('ok')}, result_count={call.get('result_count')})"
        for entry in trace
        for call in (entry.get("tool_calls") or [])
    ]

    print(f"Ticket:         {ticket_id}  (customer {external_user_id}, channel {channel})")
    print(f"Ticket text:    {text!r}")
    print(f"Classification: category={cls.get('category')}  urgency={cls.get('urgency')}  "
          f"hard_escalate={cls.get('hard_escalate')}")
    print(f"Route:          {' -> '.join(routes) if routes else '(none)'}")
    print(f"Confidence:     {result.get('confidence')}")
    print(f"Tools invoked:  {', '.join(tool_calls) if tool_calls else '(none)'}")
    print(f"Final status:   {result['final_status']}")
    print(f"Assistant:      {result['messages'][-1].content}")
    print("-" * 100)
    return result

#### Scenario 1 — normal FAQ resolution

In [5]:
_ = run_ticket("demo-faq-1", "cultpass", "f556c0", "What's included in a CultPass subscription?")

Ticket:         demo-faq-1  (customer f556c0, channel chat)
Ticket text:    "What's included in a CultPass subscription?"
Classification: category=general  urgency=low  hard_escalate=False
Route:          general_resolver -> finalize
Confidence:     0.765
Tools invoked:  search_knowledge_base(ok=True, result_count=3)
Final status:   resolved
Assistant:      A CultPass subscription includes 4 cultural experiences per month, which can be art exhibitions, museum entries, music concerts, film screenings, and more. Some premium experiences may require an additional fee, which will be visible in the app during reservation.
----------------------------------------------------------------------------------------------------


#### Scenario 2 — urgent / complex case

In [6]:
_ = run_ticket(
    "demo-urgent-1",
    "cultpass",
    "88382b",
    "I was charged twice this month for my subscription and need this fixed immediately -- "
    "my bank statement shows two separate charges!",
    reported_urgency="high",
)

Ticket:         demo-urgent-1  (customer 88382b, channel chat)
Ticket text:    'I was charged twice this month for my subscription and need this fixed immediately -- my bank statement shows two separate charges!'
Classification: category=billing  urgency=high  hard_escalate=False
Route:          billing_resolver -> finalize
Confidence:     0.8
Tools invoked:  get_subscription_status(ok=True, result_count=1), search_knowledge_base(ok=True, result_count=3)
Final status:   resolved
Assistant:      Your subscription is currently cancelled, so you shouldn't be charged again. However, if you see two charges this month, it might be due to a processing error or a previous billing cycle.

I recommend checking your payment method in the app under 'My Account' > 'Payment Methods' to ensure everything is correct. If the issue persists, please contact your bank for clarification on the charges.

If you need further assistance or want to reactivate your subscription, let me know!
-------------------

#### Scenario 3 — successful tool-driven action (booking a real reservation)

In [7]:
_ = run_ticket("demo-booking-1", "cultpass", "f556c0", "I'd like to book a spot for the Samba Night at Lapa event.")

Ticket:         demo-booking-1  (customer f556c0, channel chat)
Ticket text:    "I'd like to book a spot for the Samba Night at Lapa event."
Classification: category=booking  urgency=low  hard_escalate=False
Route:          booking_resolver -> finalize
Confidence:     1.0
Tools invoked:  search_experiences(ok=True, result_count=1), manage_reservation(ok=True, result_count=1)
Final status:   resolved
Assistant:      You've successfully booked a spot for the Samba Night at Lapa on August 17, 2026. Enjoy the night of dancing!
----------------------------------------------------------------------------------------------------


#### Scenario 4 — missing-knowledge escalation

In [8]:
_ = run_ticket(
    "demo-noinfo-1",
    "cultpass",
    "f1f10d",
    "What's your policy on transferring my CultPass membership to a family member if I permanently move abroad?",
)

Ticket:         demo-noinfo-1  (customer f1f10d, channel chat)
Ticket text:    "What's your policy on transferring my CultPass membership to a family member if I permanently move abroad?"
Classification: category=general  urgency=low  hard_escalate=False
Route:          general_resolver -> escalation
Confidence:     0.0
Tools invoked:  search_knowledge_base(ok=True, result_count=3)
Final status:   escalated
Assistant:      Thank you for reaching out to us. I've passed your request regarding the transfer of your CultPass membership to a family member to a specialist who will be able to assist you further. We appreciate your patience as we look into this for you.
----------------------------------------------------------------------------------------------------


#### Scenario 5 — hard-escalate bypass (blocked account)

In [9]:
_ = run_ticket(
    "demo-blocked-1",
    "cultpass",
    "a4ab87",
    "I can't log in and it says my account is blocked. This is urgent, please help.",
    reported_urgency="high",
)

Ticket:         demo-blocked-1  (customer a4ab87, channel chat)
Ticket text:    "I can't log in and it says my account is blocked. This is urgent, please help."
Classification: category=account  urgency=high  hard_escalate=True
Route:          escalation
Confidence:     None
Tools invoked:  (none)
Final status:   escalated
Assistant:      Thank you for reaching out. I understand that you're experiencing an urgent issue with your account being blocked. I've passed your request to a specialist who will be able to assist you further.
----------------------------------------------------------------------------------------------------


### Short-term (session) memory: a follow-up turn that depends on the first

Two turns, the same `ticket_id` (= `thread_id`). The second message only
makes sense in light of the first -- proving `state["messages"]` actually
reaches agent reasoning, not just storage.

In [10]:
_ = run_ticket(
    "demo-multiturn-1", "cultpass", "e6376d",
    "I'm on the premium plan and I think I was double-charged this month.",
)
r2 = run_ticket(
    "demo-multiturn-1", "cultpass", "e6376d",
    "Can you go ahead and fix the issue we just discussed?",
)

Ticket:         demo-multiturn-1  (customer e6376d, channel chat)
Ticket text:    "I'm on the premium plan and I think I was double-charged this month."
Classification: category=billing  urgency=medium  hard_escalate=False
Route:          billing_resolver -> finalize
Confidence:     0.9
Tools invoked:  get_subscription_status(ok=True, result_count=1), recall_customer_memory(ok=True, result_count=0)
Final status:   resolved
Assistant:      You're currently on the basic plan, not the premium plan. If you believe you were double-charged, I recommend checking your payment history for any discrepancies. If you need further assistance with this issue, let me know!
----------------------------------------------------------------------------------------------------


Ticket:         demo-multiturn-1  (customer e6376d, channel chat)
Ticket text:    'Can you go ahead and fix the issue we just discussed?'
Classification: category=billing  urgency=low  hard_escalate=False
Route:          billing_resolver -> finalize -> billing_resolver -> finalize
Confidence:     0.75
Tools invoked:  get_subscription_status(ok=True, result_count=1), recall_customer_memory(ok=True, result_count=0)
Final status:   resolved
Assistant:      Since you're currently on the basic plan, I can't change your subscription to premium without your confirmation. Would you like me to upgrade your subscription to the premium plan?
----------------------------------------------------------------------------------------------------


In [11]:
history = list(orchestrator.get_state_history(config={"configurable": {"thread_id": "demo-multiturn-1"}}))
print(f"Checkpoints recorded for this thread_id (LangGraph's own get_state_history()): {len(history)}\n")

# history[0] is the most recent checkpoint -- the full, accumulated message
# history for this session, not just this turn's delta.
latest_messages = history[0].values["messages"]
print("Full session message history, from get_state_history()[0].values['messages']:\n")
for m in latest_messages:
    print(f"  [{type(m).__name__}] {m.content}")

Checkpoints recorded for this thread_id (LangGraph's own get_state_history()): 16

Full session message history, from get_state_history()[0].values['messages']:

  [HumanMessage] I'm on the premium plan and I think I was double-charged this month.
  [AIMessage] You're currently on the basic plan, not the premium plan. If you believe you were double-charged, I recommend checking your payment history for any discrepancies. If you need further assistance with this issue, let me know!
  [HumanMessage] Can you go ahead and fix the issue we just discussed?
  [AIMessage] Since you're currently on the basic plan, I can't change your subscription to premium without your confirmation. Would you like me to upgrade your subscription to the premium plan?


### Long-term (cross-session) memory: a preference saved on one ticket, recalled on another

Session A states a preference; Session B is a *brand-new ticket/thread* for
the same customer. Context Loader recalls the preference from
`customer_memory` before the Classifier even runs, proving it survives
across sessions, not just within one.

In [12]:
_ = run_ticket(
    "demo-prefs-1", "cultpass", "f1f10d",
    "Please only contact me by email from now on -- I don't want phone calls.",
)

Ticket:         demo-prefs-1  (customer f1f10d, channel chat)
Ticket text:    "Please only contact me by email from now on -- I don't want phone calls."
Classification: category=account  urgency=low  hard_escalate=False
Route:          account_resolver -> finalize
Confidence:     0.85
Tools invoked:  recall_customer_memory(ok=True, result_count=0), get_customer_profile(ok=True, result_count=1), search_knowledge_base(ok=True, result_count=3)
Final status:   resolved
Assistant:      I've noted your preference to only be contacted by email from now on. 

For managing your notification preferences, you can adjust these in the app under 'Settings' > 'Notifications'. This allows you to toggle different types of notifications, including email alerts for subscription renewals and receipts.
----------------------------------------------------------------------------------------------------


In [13]:
r_prefs2 = run_ticket("demo-prefs-2", "cultpass", "f1f10d", "How do I check my current subscription tier?")

print("Long-term memories recalled on this brand-new session (different ticket_id/thread_id):\n")
for m in r_prefs2["user_context"]["long_term_memories"]:
    print(f"  [{m.get('memory_type')}] {m.get('content')}")

Ticket:         demo-prefs-2  (customer f1f10d, channel chat)
Ticket text:    'How do I check my current subscription tier?'
Classification: category=billing  urgency=low  hard_escalate=False
Route:          billing_resolver -> finalize
Confidence:     0.9
Tools invoked:  (none)
Final status:   resolved
Assistant:      You can check your current subscription tier by looking in the app under 'Account' or 'Subscription' settings. Since I have access to your account, I can confirm that your current subscription tier is "basic," but if you need to check it yourself, that's where to find it!
----------------------------------------------------------------------------------------------------
Long-term memories recalled on this brand-new session (different ticket_id/thread_id):

  [resolution_summary] Resolved ticket demo-prefs-1: I've noted your preference to only be contacted by email from now on. 

For managing your notification preferences, you can adjust these in the app under 'Settings'

### Retrieval / escalation / tool-usage metrics

Computed by `agentic/trace_metrics.py` directly from `logs/uda_hub_trace.jsonl`
-- the same redacted, safe-metadata-only log every scenario above just wrote to.

In [14]:
from agentic.trace_metrics import format_report, report

print(format_report(report()))

Tickets processed: 9 (7 resolved, 2 escalated)
Escalation frequency: 22.2%
Knowledge-retrieval success rate: 100.0% (4/4 searches found a relevant article)
Tool usage:
  get_customer_profile: 1 calls, 100.0% success rate
  get_subscription_status: 2 calls, 100.0% success rate
  manage_reservation: 1 calls, 100.0% success rate
  recall_customer_memory: 2 calls, 100.0% success rate
  search_experiences: 1 calls, 100.0% success rate
  search_knowledge_base: 4 calls, 100.0% success rate
